# MNIST Neural Network: Intermediate Level 1
## Layer-Based Object-Oriented Design

**Complexity Level**: Intermediate  
**Architectural Style**: Layer as primary abstraction  
**Focus**: Separating concerns while keeping design manageable

---

## The Problem: Handwritten Digit Recognition

The MNIST dataset presents a classic machine learning challenge: given an image of a handwritten digit, determine which digit (0-9) it represents.

**Inputs**: Each image is a 28×28 pixel grayscale image, which we flatten into a single list of 784 values. Each pixel value ranges from 0 (black) to 255 (white), though we typically normalise these to the range 0.0 to 1.0 by dividing by 255. So our neural network receives 784 floating-point numbers as input for each image.

**Outputs**: The network produces 10 values, one for each possible digit (0 through 9). These outputs represent the network's confidence that the input image shows each respective digit. We typically interpret these as probabilities (summing to 1.0 after applying softmax), and the digit with the highest probability becomes our prediction. For training, we compare these outputs against "one-hot" encoded labels where the correct digit has value 1.0 and all others have 0.0.

**The Task**: Build a function that maps 784 input values to 10 output values, learning the patterns that distinguish each digit through repeated exposure to labelled examples.

---

## Why This Approach?

This notebook uses **Layer as the primary abstraction**. Each layer:
- Owns its weights and biases
- Knows how to compute forward pass
- Knows how to compute backward pass
- Can update its own parameters

The Network class simply orchestrates layers in sequence.

**Design Philosophy**: "A layer is a transformation with learnable parameters."

**Trade-offs**:
- Cleaner than monolith, simpler than full decomposition
- Layers are self-contained and testable
- Less flexible than having separate Node/Connection objects
- Activation functions are simple functions, not separate classes

**Alternative**: Could use abstract Layer base class with DenseLayer, ConvLayer subclasses (see advanced notebook).

In [ ]:
# Only standard library
import math
import random
from typing import List, Tuple, Callable, Optional

random.seed(42)

---

## Part 1: Activation Functions Module

We define activation functions as simple functions (not classes). This is a **design choice** that prioritises simplicity.

**Alternative**: Strategy Pattern with `ActivationFunction` base class (see original notebook). That approach allows runtime inspection and is more extensible, but adds complexity.

In [ ]:
# ==================== Activation Functions ====================
# Design choice: Simple functions instead of classes.
# Each function takes a single value and returns the transformed value.
# Derivatives are separate functions following naming convention: {name}_derivative

def sigmoid(x: float) -> float:
    """Sigmoid: squashes to (0, 1). Good for output probabilities."""
    x = max(-500, min(500, x))
    return 1.0 / (1.0 + math.exp(-x))

def sigmoid_derivative(x: float) -> float:
    """Derivative: sigmoid(x) * (1 - sigmoid(x))."""
    s = sigmoid(x)
    return s * (1.0 - s)

def relu(x: float) -> float:
    """ReLU: max(0, x). Fast, helps with vanishing gradients."""
    return max(0.0, x)

def relu_derivative(x: float) -> float:
    """Derivative: 1 if x > 0, else 0."""
    return 1.0 if x > 0 else 0.0

def leaky_relu(x: float, alpha: float = 0.01) -> float:
    """
    Leaky ReLU: x if x > 0, else alpha * x.
    
    Alternative to ReLU that allows small gradients for negative inputs.
    Helps prevent 'dying ReLU' problem.
    """
    return x if x > 0 else alpha * x

def leaky_relu_derivative(x: float, alpha: float = 0.01) -> float:
    """Derivative: 1 if x > 0, else alpha."""
    return 1.0 if x > 0 else alpha

def softmax(values: List[float]) -> List[float]:
    """Softmax: converts logits to probability distribution."""
    max_val = max(values)
    exp_values = [math.exp(v - max_val) for v in values]
    total = sum(exp_values)
    return [ev / total for ev in exp_values]

# Registry for easy lookup
ACTIVATIONS = {
    'sigmoid': (sigmoid, sigmoid_derivative),
    'relu': (relu, relu_derivative),
    'leaky_relu': (leaky_relu, leaky_relu_derivative),
}

def get_activation(name: str) -> Tuple[Callable, Callable]:
    """
    Get activation function and derivative by name.
    
    Design choice: Registry pattern for extensibility.
    Alternative: If-else chain (simpler but less extensible).
    """
    if name not in ACTIVATIONS:
        raise ValueError(f"Unknown activation: {name}. Options: {list(ACTIVATIONS.keys())}")
    return ACTIVATIONS[name]

---

## Part 2: The Layer Class

The `Layer` class is our central abstraction. Each layer:
- Stores weights connecting from previous layer
- Stores biases for its nodes
- Computes forward and backward passes
- Updates its own parameters

**Design Choice**: Weights stored as 2D list `[input_idx][output_idx]`.

**Alternative**: Store as `[output_idx][input_idx]` (matches some textbooks). Our choice makes the forward pass more intuitive.

In [ ]:
class Layer:
    """
    A single layer in a neural network.
    
    Each layer represents a transformation: y = activation(W*x + b)
    where W is the weight matrix, b is the bias vector, and x is input.
    
    The layer owns its parameters and knows how to:
    - Compute output given input (forward)
    - Compute gradients given upstream error (backward)
    - Update its parameters
    
    Attributes:
        input_size: Number of inputs this layer receives
        output_size: Number of outputs this layer produces
        activation_name: Which activation function to use
        weights: 2D list [input_idx][output_idx]
        biases: 1D list [output_idx]
    """
    
    def __init__(self, 
                 input_size: int, 
                 output_size: int, 
                 activation: str = 'relu',
                 weight_init: str = 'xavier'):
        """
        Initialise layer with given dimensions.
        
        Args:
            input_size: Dimension of input vector
            output_size: Dimension of output vector (number of neurons)
            activation: 'relu', 'sigmoid', or 'leaky_relu'
            weight_init: 'xavier', 'he', or 'random'
        
        Weight initialisation options:
        - xavier: Good for sigmoid/tanh, limit = sqrt(6/(fan_in + fan_out))
        - he: Good for ReLU, limit = sqrt(2/fan_in)
        - random: Simple uniform [-0.5, 0.5]
        """
        self.input_size = input_size
        self.output_size = output_size
        self.activation_name = activation
        self.activate, self.activate_derivative = get_activation(activation)
        
        # Initialise weights
        self.weights = self._init_weights(weight_init)
        self.biases = [0.0] * output_size
        
        # Cache for backpropagation (filled during forward pass)
        self.last_input = None
        self.last_z = None  # Pre-activation values
        self.last_output = None
    
    def _init_weights(self, method: str) -> List[List[float]]:
        """
        Initialise weight matrix using specified method.
        
        Returns:
            2D list of weights [input_size][output_size]
        """
        if method == 'xavier':
            limit = math.sqrt(6.0 / (self.input_size + self.output_size))
        elif method == 'he':
            limit = math.sqrt(2.0 / self.input_size)
        else:  # random
            limit = 0.5
        
        return [
            [random.uniform(-limit, limit) for _ in range(self.output_size)]
            for _ in range(self.input_size)
        ]
    
    def forward(self, inputs: List[float]) -> List[float]:
        """
        Compute layer output: activation(W*x + b).
        
        Args:
            inputs: Input vector of length input_size
        
        Returns:
            Output vector of length output_size
        
        Side effects:
            Caches values needed for backpropagation.
        """
        if len(inputs) != self.input_size:
            raise ValueError(f"Expected {self.input_size} inputs, got {len(inputs)}")
        
        self.last_input = inputs
        
        # Compute z = W*x + b (pre-activation)
        z = []
        for j in range(self.output_size):
            total = self.biases[j]
            for i in range(self.input_size):
                total += inputs[i] * self.weights[i][j]
            z.append(total)
        
        self.last_z = z
        
        # Apply activation
        output = [self.activate(val) for val in z]
        self.last_output = output
        
        return output
    
    def backward(self, output_delta: List[float]) -> List[float]:
        """
        Compute gradients and return delta for previous layer.
        
        Args:
            output_delta: Error signal from next layer (or loss gradient)
        
        Returns:
            input_delta: Error signal to pass to previous layer
        
        Side effects:
            Stores weight_grads and bias_grads for later update.
        """
        # Compute gradients for weights and biases
        self.weight_grads = [
            [self.last_input[i] * output_delta[j] for j in range(self.output_size)]
            for i in range(self.input_size)
        ]
        self.bias_grads = output_delta[:]
        
        # Compute delta for previous layer
        input_delta = []
        for i in range(self.input_size):
            error_sum = sum(
                output_delta[j] * self.weights[i][j]
                for j in range(self.output_size)
            )
            # Multiply by activation derivative of input
            # Note: we need the pre-activation value of the previous layer
            # Since we don't have it, we use the input value and assume
            # the activation derivative is handled by the previous layer
            input_delta.append(error_sum)
        
        return input_delta
    
    def update(self, learning_rate: float):
        """
        Update weights and biases using stored gradients.
        
        Args:
            learning_rate: Step size for gradient descent
        """
        for i in range(self.input_size):
            for j in range(self.output_size):
                self.weights[i][j] -= learning_rate * self.weight_grads[i][j]
        
        for j in range(self.output_size):
            self.biases[j] -= learning_rate * self.bias_grads[j]
    
    def num_parameters(self) -> int:
        """Return total number of trainable parameters."""
        return self.input_size * self.output_size + self.output_size
    
    def __repr__(self):
        return f"Layer({self.input_size} -> {self.output_size}, {self.activation_name})"

---

## Part 3: Output Layer

The output layer is special: it uses softmax activation and has a different backward pass (combined with cross-entropy loss for numerical stability).

**Design Choice**: Separate class inheriting from Layer.

**Alternative**: Single Layer class with `is_output` flag (simpler but less clean).

In [ ]:
class OutputLayer(Layer):
    """
    Output layer with softmax activation.
    
    This layer is specialised for classification:
    - Uses softmax (not pluggable activation)
    - Has simplified backward pass when paired with cross-entropy
    
    The softmax + cross-entropy combination has a beautiful property:
    the gradient simplifies to (output - target), avoiding numerical issues.
    """
    
    def __init__(self, input_size: int, num_classes: int, weight_init: str = 'xavier'):
        """
        Initialise output layer.
        
        Args:
            input_size: Size of input from previous layer
            num_classes: Number of output classes (10 for MNIST)
            weight_init: Weight initialisation method
        """
        # Don't call parent __init__ fully - we override activation
        self.input_size = input_size
        self.output_size = num_classes
        self.activation_name = 'softmax'
        
        # Initialise weights using parent's method
        if weight_init == 'xavier':
            limit = math.sqrt(6.0 / (input_size + num_classes))
        elif weight_init == 'he':
            limit = math.sqrt(2.0 / input_size)
        else:
            limit = 0.5
        
        self.weights = [
            [random.uniform(-limit, limit) for _ in range(num_classes)]
            for _ in range(input_size)
        ]
        self.biases = [0.0] * num_classes
        
        # Cache
        self.last_input = None
        self.last_z = None
        self.last_output = None
    
    def forward(self, inputs: List[float]) -> List[float]:
        """
        Compute output probabilities using softmax.
        """
        self.last_input = inputs
        
        # Compute z = W*x + b
        z = []
        for j in range(self.output_size):
            total = self.biases[j]
            for i in range(self.input_size):
                total += inputs[i] * self.weights[i][j]
            z.append(total)
        
        self.last_z = z
        
        # Softmax activation
        output = softmax(z)
        self.last_output = output
        
        return output
    
    def backward_with_target(self, target: List[float]) -> List[float]:
        """
        Compute gradients using target directly.
        
        For softmax + cross-entropy, gradient = output - target.
        This is more numerically stable than computing separately.
        
        Args:
            target: One-hot encoded true label
        
        Returns:
            Delta to pass to previous layer
        """
        # Softmax + cross-entropy gradient
        output_delta = [
            self.last_output[j] - target[j] 
            for j in range(self.output_size)
        ]
        
        # Compute gradients
        self.weight_grads = [
            [self.last_input[i] * output_delta[j] for j in range(self.output_size)]
            for i in range(self.input_size)
        ]
        self.bias_grads = output_delta[:]
        
        # Delta for previous layer
        input_delta = []
        for i in range(self.input_size):
            error_sum = sum(
                output_delta[j] * self.weights[i][j]
                for j in range(self.output_size)
            )
            input_delta.append(error_sum)
        
        return input_delta
    
    def predict(self) -> int:
        """Return predicted class (highest probability)."""
        return self.last_output.index(max(self.last_output))

---

## Part 4: The Network Class

The Network class orchestrates layers but contains minimal logic itself.

**Design Principle**: Network is a coordinator, not a worker.

In [ ]:
class Network:
    """
    A feedforward neural network composed of layers.
    
    The Network's job is coordination:
    - Chain layers together
    - Orchestrate forward/backward passes
    - Manage training loop
    
    Actual computation is delegated to Layer objects.
    """
    
    def __init__(self, layers: List[Layer]):
        """
        Construct network from list of layers.
        
        Args:
            layers: List of Layer objects (last should be OutputLayer)
        
        Design choice: Accept pre-constructed layers.
        Alternative: Accept architecture spec and construct layers internally.
        """
        self.layers = layers
        self._validate_layers()
    
    def _validate_layers(self):
        """Check that layers are compatible."""
        for i in range(len(self.layers) - 1):
            if self.layers[i].output_size != self.layers[i+1].input_size:
                raise ValueError(
                    f"Layer {i} output ({self.layers[i].output_size}) "
                    f"doesn't match layer {i+1} input ({self.layers[i+1].input_size})"
                )
    
    def forward(self, inputs: List[float]) -> List[float]:
        """
        Forward pass through all layers.
        """
        current = inputs
        for layer in self.layers:
            current = layer.forward(current)
        return current
    
    def backward(self, target: List[float], learning_rate: float):
        """
        Backward pass and parameter update.
        
        Args:
            target: One-hot encoded label
            learning_rate: Learning rate for updates
        """
        # Output layer uses special backward
        output_layer = self.layers[-1]
        if isinstance(output_layer, OutputLayer):
            delta = output_layer.backward_with_target(target)
        else:
            raise TypeError("Last layer must be OutputLayer")
        
        # Backward through hidden layers
        for i in range(len(self.layers) - 2, -1, -1):
            layer = self.layers[i]
            # Apply activation derivative
            delta_with_activation = [
                delta[j] * layer.activate_derivative(layer.last_z[j])
                for j in range(layer.output_size)
            ]
            delta = layer.backward(delta_with_activation)
        
        # Update all layers
        for layer in self.layers:
            layer.update(learning_rate)
    
    def predict(self, inputs: List[float]) -> int:
        """Make prediction (class index)."""
        self.forward(inputs)
        return self.layers[-1].predict()
    
    def predict_proba(self, inputs: List[float]) -> List[float]:
        """Get prediction probabilities."""
        return self.forward(inputs)
    
    def summary(self):
        """Print network summary."""
        print("Network Summary")
        print("=" * 50)
        total_params = 0
        for i, layer in enumerate(self.layers):
            params = layer.num_parameters()
            total_params += params
            print(f"Layer {i}: {layer} ({params:,} params)")
        print("=" * 50)
        print(f"Total parameters: {total_params:,}")

---

## Part 5: Trainer Class

Separating training logic from the network itself follows Single Responsibility Principle.

**Design Choice**: Trainer is stateless - it receives network and data, does training.

**Alternative**: Trainer could maintain state (momentum, learning rate schedules).

In [ ]:
class Trainer:
    """
    Handles network training.
    
    Separated from Network to allow different training strategies
    without modifying the Network class.
    
    Future extensions (not implemented here):
    - Learning rate scheduling
    - Early stopping
    - Momentum / Adam optimisation
    - Batch training
    """
    
    def __init__(self, network: Network, learning_rate: float = 0.01):
        """
        Initialise trainer.
        
        Args:
            network: Network to train
            learning_rate: Step size for gradient descent
        """
        self.network = network
        self.learning_rate = learning_rate
    
    @staticmethod
    def _one_hot(label: int, num_classes: int) -> List[float]:
        """Convert integer label to one-hot vector."""
        vec = [0.0] * num_classes
        vec[label] = 1.0
        return vec
    
    @staticmethod
    def _cross_entropy(predicted: List[float], target: List[float]) -> float:
        """Compute cross-entropy loss."""
        epsilon = 1e-15
        return -sum(
            t * math.log(max(p, epsilon))
            for p, t in zip(predicted, target)
            if t > 0
        )
    
    def train_step(self, inputs: List[float], label: int) -> Tuple[float, bool]:
        """
        Train on single example.
        
        Returns:
            loss: Cross-entropy loss
            correct: Whether prediction was correct
        """
        num_classes = self.network.layers[-1].output_size
        target = self._one_hot(label, num_classes)
        
        # Forward
        output = self.network.forward(inputs)
        loss = self._cross_entropy(output, target)
        predicted = output.index(max(output))
        
        # Backward and update
        self.network.backward(target, self.learning_rate)
        
        return loss, (predicted == label)
    
    def train_epoch(self, 
                    data: List[List[float]], 
                    labels: List[int]) -> Tuple[float, float]:
        """
        Train on entire dataset once.
        
        Returns:
            avg_loss: Average loss over dataset
            accuracy: Fraction correct
        """
        total_loss = 0.0
        correct = 0
        
        for inputs, label in zip(data, labels):
            loss, is_correct = self.train_step(inputs, label)
            total_loss += loss
            correct += int(is_correct)
        
        return total_loss / len(data), correct / len(data)
    
    def fit(self, 
            train_data: List[List[float]], 
            train_labels: List[int],
            epochs: int = 10,
            val_data: Optional[List[List[float]]] = None,
            val_labels: Optional[List[int]] = None,
            verbose: bool = True) -> dict:
        """
        Train for multiple epochs.
        
        Args:
            train_data, train_labels: Training set
            epochs: Number of epochs
            val_data, val_labels: Optional validation set
            verbose: Print progress
        
        Returns:
            history: Dict with loss/accuracy per epoch
        """
        history = {
            'train_loss': [], 'train_acc': [],
            'val_loss': [], 'val_acc': []
        }
        
        for epoch in range(epochs):
            loss, acc = self.train_epoch(train_data, train_labels)
            history['train_loss'].append(loss)
            history['train_acc'].append(acc)
            
            if val_data is not None:
                val_acc = self.evaluate(val_data, val_labels)
                history['val_acc'].append(val_acc)
            
            if verbose and ((epoch + 1) % 5 == 0 or epoch == 0):
                msg = f"Epoch {epoch+1:3d}: Loss={loss:.4f}, Acc={acc:.3f}"
                if val_data is not None:
                    msg += f", Val Acc={val_acc:.3f}"
                print(msg)
        
        return history
    
    def evaluate(self, data: List[List[float]], labels: List[int]) -> float:
        """Evaluate accuracy on dataset."""
        correct = sum(
            1 for inputs, label in zip(data, labels)
            if self.network.predict(inputs) == label
        )
        return correct / len(data)

---

## Part 6: Putting It Together

Now we can build and train a network using our layer-based design.

In [ ]:
# Data generation helper
def generate_data(num_samples: int, input_size: int, num_classes: int):
    """Generate synthetic classification data."""
    data = []
    labels = []
    chunk_size = input_size // num_classes
    
    for _ in range(num_samples):
        inputs = [random.random() for _ in range(input_size)]
        chunk_sums = [
            sum(inputs[c * chunk_size:(c + 1) * chunk_size])
            for c in range(num_classes)
        ]
        label = chunk_sums.index(max(chunk_sums))
        data.append(inputs)
        labels.append(label)
    
    return data, labels

In [ ]:
# Configuration
INPUT_SIZE = 20
HIDDEN_SIZE = 16
NUM_CLASSES = 4

# Build network by composing layers
layers = [
    Layer(INPUT_SIZE, HIDDEN_SIZE, activation='relu', weight_init='he'),
    OutputLayer(HIDDEN_SIZE, NUM_CLASSES, weight_init='xavier')
]

network = Network(layers)
network.summary()

In [ ]:
# Generate data
train_data, train_labels = generate_data(500, INPUT_SIZE, NUM_CLASSES)
test_data, test_labels = generate_data(100, INPUT_SIZE, NUM_CLASSES)

print(f"Training samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")

In [ ]:
# Train
trainer = Trainer(network, learning_rate=0.1)

print("Training...\n")
history = trainer.fit(
    train_data, train_labels,
    epochs=20,
    val_data=test_data,
    val_labels=test_labels
)

print(f"\nFinal test accuracy: {trainer.evaluate(test_data, test_labels):.3f}")

In [ ]:
# Easy to try different architectures
print("\nComparing architectures:")
print("=" * 60)

architectures = [
    ([INPUT_SIZE, 8, NUM_CLASSES], "Small"),
    ([INPUT_SIZE, 16, NUM_CLASSES], "Medium"),
    ([INPUT_SIZE, 16, 8, NUM_CLASSES], "Two hidden"),
]

for arch, name in architectures:
    # Build layers
    layers = []
    for i in range(len(arch) - 2):
        layers.append(Layer(arch[i], arch[i+1], activation='relu'))
    layers.append(OutputLayer(arch[-2], arch[-1]))
    
    net = Network(layers)
    trainer = Trainer(net, learning_rate=0.1)
    trainer.fit(train_data, train_labels, epochs=15, verbose=False)
    
    test_acc = trainer.evaluate(test_data, test_labels)
    arch_str = ' -> '.join(map(str, arch))
    print(f"{name:12s} ({arch_str}): Test acc = {test_acc:.3f}")

---

## Summary

### Architecture Diagram

```
┌─────────────┐     ┌─────────────┐
│   Trainer   │────>│   Network   │
└─────────────┘     └──────┬──────┘
                           │
                    ┌──────┴──────┐
                    │   [layers]  │
                    └──────┬──────┘
              ┌────────────┼────────────┐
              │            │            │
        ┌─────┴─────┐┌─────┴─────┐┌─────┴─────┐
        │   Layer   ││   Layer   ││OutputLayer│
        │(hidden 1) ││(hidden 2) ││ (softmax) │
        └───────────┘└───────────┘└───────────┘
```

### Design Choices Summary

| Choice | Alternative | Rationale |
|--------|-------------|----------|
| Layer owns weights | Separate Connection class | Simpler, weights are layer's concern |
| Functions for activations | Strategy Pattern classes | Less overhead, sufficient flexibility |
| Separate OutputLayer | is_output flag | Cleaner separation of softmax logic |
| Separate Trainer class | Training in Network | Single Responsibility Principle |
| Accept layer list | Architecture dict | More flexible composition |

### For MNIST

```python
layers = [
    Layer(784, 128, activation='relu'),
    Layer(128, 64, activation='relu'),
    OutputLayer(64, 10)
]
network = Network(layers)
trainer = Trainer(network, learning_rate=0.01)
trainer.fit(train_images, train_labels, epochs=10)
```

---

**Next notebook**: See how we can add more abstraction with a component-based design.